# Context Assembly and Window Management

Retrieval can return ten strong chunks — but the LLM may only **fit four** in its context window. **Context assembly** is the post-retrieval stage that decides **which** chunks to include, in **what order**, and **what to drop** when the token budget is full. Poor assembly causes wrong answers even when the right document was retrieved — especially the **lost in the middle** effect, where models under-use evidence buried in the center of long prompts.

Notebook **05** showed how to **inject** context into prompts. This notebook focuses on **packing** that context under a finite budget: token math, ordering strategies, deduplication, compression, map-reduce for large evidence sets, dynamic $k$, and utilization logging.

Embedding and chunking mechanics live in **05. Embeddings & Vector DBs** (**04** chunking). Advanced ranking (hybrid, rerank) is in **07**. Here we assume you already have a ranked list of chunks — and need to fit them into the window.

Prerequisites: **05. Prompting and Context Injection for RAG**, **02.07 Tokens and Context Windows**, **02. Naive RAG Pipeline**. Next: **07. Advanced Retrieval for RAG**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

try:
    import tiktoken

    enc = tiktoken.encoding_for_model("gpt-4o-mini")

    def count_tokens(text: str) -> int:
        return len(enc.encode(text))

    TOKENIZER = "tiktoken (gpt-4o-mini)"
except Exception:
    def count_tokens(text: str) -> int:
        return max(1, len(text) // 4)  # rough fallback

    TOKENIZER = "char//4 heuristic"

print("Token counter:", TOKENIZER)

---

## 1. Why Assembly Is a Separate Stage

Naive RAG (**02**) often does `retrieve(k=3)` and stuffs all results into the prompt. That breaks down when:

| Situation | Problem |
|-----------|--------|
| Large $k$ or big chunks | Exceeds context window → silent truncation |
| Chat history present | Less room for evidence (**11**) |
| Long system + few-shot prompts | Context squeezed (**05**) |
| Duplicate chunks (c2 + c5) | Wasted tokens on Alembic overlap |

```
Retrieve (k=20) ──► Rank / dedupe ──► Assemble (token budget) ──► Prompt (**05**)
                         ▲                    ▲
                    notebook 07         this notebook
```

**Retrieve wide, assemble narrow** — fetch more candidates than you can fit, then pack the best subset.

---

## 2. Context Budget Equation

Every chat request shares one finite window $C_{\max}$:

$$T_{\text{available}} = C_{\max} - T_{\text{system}} - T_{\text{history}} - T_{\text{question}} - T_{\text{reserved\_output}}$$

| Term | Typical content | Notes |
|------|-----------------|-------|
| $C_{\max}$ | Model limit (e.g. 128k) | Hard cap |
| $T_{\text{system}}$ | Grounding rules (**05**) | 100–500 tokens |
| $T_{\text{history}}$ | Prior chat turns (**11**) | Grows each turn |
| $T_{\text{question}}$ | Latest user question | Usually small |
| $T_{\text{reserved\_output}}$ | Headroom for answer | 256–1024 tokens |
| **Remaining** | **For retrieved context** | Often 2k–8k in prod |

### 2.1 Rules of Thumb

- Never target **100%** utilization — tokenizer drift and API overhead exist
- Reserve output tokens **before** packing chunks
- Log `usage.prompt_tokens` from the API to calibrate estimates

In [ ]:
C_MAX = 8192
T_SYSTEM = 220
T_HISTORY = 450
T_QUESTION = 35
T_OUTPUT_RESERVE = 512

T_AVAILABLE = C_MAX - T_SYSTEM - T_HISTORY - T_QUESTION - T_OUTPUT_RESERVE
print(f"Model window:     {C_MAX}")
print(f"Context budget:   {T_AVAILABLE} tokens for retrieved chunks")

---

## 3. Demo Corpus — Retrieved Candidates

Simulated retrieval results for the shared **c1–c5** knowledge base with relevance scores (as if returned from vector search + reranker):

In [ ]:
CANDIDATES = [
    {
        "id": "c3",
        "source": "security-docs",
        "score": 0.92,
        "text": (
            "JWT bearer tokens authenticate API requests. "
            "Clients send Authorization: Bearer token header."
        ),
    },
    {
        "id": "c2",
        "source": "db-docs",
        "score": 0.88,
        "text": (
            "Alembic applies SQLAlchemy schema migrations. "
            "Run alembic upgrade head after pulling new revisions."
        ),
    },
    {
        "id": "c5",
        "source": "db-docs",
        "score": 0.85,
        "text": (
            "Alembic autogenerate compares SQLAlchemy models to the live database schema "
            "and drafts revision files."
        ),
    },
    {
        "id": "c1",
        "source": "api-docs",
        "score": 0.81,
        "text": (
            "The Notes API is built with FastAPI and stores notes in memory for demos. "
            "Routes live under /notes."
        ),
    },
    {
        "id": "c4",
        "source": "test-docs",
        "score": 0.79,
        "text": (
            "Pytest fixtures share database setup across tests. "
            "Use conftest.py for session-scoped engines."
        ),
    },
]

for c in CANDIDATES:
    c["tokens"] = count_tokens(c["text"]) + 12  # delimiter overhead per chunk
    print(f"{c['id']}  score={c['score']:.2f}  tokens≈{c['tokens']}")

---

## 4. Greedy Packing by Token Budget

The simplest assembler walks ranked chunks and adds each until the budget is full.

```python
packed, used = [], 0
for chunk in ranked_chunks:
    if used + chunk.tokens > budget:
        continue  # or break
    packed.append(chunk)
    used += chunk.tokens
```

| Variant | Behavior |
|---------|----------|
| **Break on first overflow** | Fixed prefix of ranking |
| **Skip and continue** | May fit a smaller later chunk |
| **Knapsack (optimal)** | Maximize score under budget — heavier compute |

In [ ]:
def greedy_pack(candidates: list[dict], budget: int) -> tuple[list[dict], int]:
    ranked = sorted(candidates, key=lambda c: c["score"], reverse=True)
    packed, used = [], 0
    for chunk in ranked:
        if used + chunk["tokens"] <= budget:
            packed.append(chunk)
            used += chunk["tokens"]
    return packed, used


BUDGET = 120  # tight budget for demonstration
packed, used = greedy_pack(CANDIDATES, BUDGET)
print(f"Budget={BUDGET}, used={used}, packed={[c['id'] for c in packed]}")

Under a tight budget, lower-ranked chunks drop off — even if they contain useful nuance. That is why **compression** and **better ranking** matter.

---

## 5. Ordering Strategies

After selecting chunks, **order** affects how the model attends to them.

| Strategy | When to use |
|----------|-------------|
| **Relevance score** | Default — highest score first |
| **Sandwich** | Best chunk at **start** and **end**; weaker in middle |
| **Recency** | Sort by `updated_at` metadata for changelogs |
| **Source diversity** | One chunk per `source` before repeats |
| **Narrative** | Original document section order |

### 5.1 Lost in the Middle

Research on long-context models shows **U-shaped attention**: tokens at the **beginning and end** of the prompt often influence the answer more than tokens in the **middle**. Sandwich ordering mitigates this by placing the strongest evidence at both ends.

In [ ]:
def sandwich_order(chunks: list[dict]) -> list[dict]:
    """Place best chunk first, second-best last, fill middle with rest."""
    if len(chunks) <= 2:
        return chunks
    ranked = sorted(chunks, key=lambda c: c["score"], reverse=True)
    middle = ranked[1:-1]
    return [ranked[0]] + middle + [ranked[-1]]


packed, _ = greedy_pack(CANDIDATES, budget=200)
linear = [c["id"] for c in packed]
sandwiched = [c["id"] for c in sandwich_order(packed)]
print("Linear by score: ", linear)
print("Sandwich order:  ", sandwiched)

In [ ]:
# Illustrative U-shaped attention proxy across chunk positions
positions = np.arange(1, 11)
attention_proxy = np.exp(-0.5 * ((positions - 5.5) / 2.2) ** 2)
attention_proxy[[0, -1]] *= 1.45
attention_proxy /= attention_proxy.max()

plt.figure(figsize=(8, 3.5))
plt.bar(positions, attention_proxy, color="steelblue", alpha=0.85)
plt.xlabel("Chunk position in assembled prompt (illustrative)")
plt.ylabel("Relative attention proxy")
plt.title("Lost in the middle — ends often dominate")
plt.tight_layout()
plt.show()

---

## 6. Deduplication Before Packing

Near-duplicate chunks waste budget. **c2** and **c5** both discuss Alembic — packing both may add little new information.

| Method | Cost | When |
|--------|------|------|
| **Exact text match** | Free | Obvious duplicates |
| **High cosine similarity** | Cheap if embeddings exist | c2/c5 overlap |
| **Same source + high overlap** | Metadata heuristic | One chunk per section |
| **LLM merge (offline)** | Expensive | Consolidate runbooks |

Deduplicate **before** token counting — not after packing.

In [ ]:
def dedupe_by_source_overlap(candidates: list[dict], max_per_source: int = 1) -> list[dict]:
    """Keep top-N chunks per source by score."""
    by_source: dict[str, list[dict]] = {}
    for c in sorted(candidates, key=lambda x: x["score"], reverse=True):
        by_source.setdefault(c["source"], []).append(c)
    out = []
    for source, chunks in by_source.items():
        out.extend(chunks[:max_per_source])
    return sorted(out, key=lambda x: x["score"], reverse=True)


deduped = dedupe_by_source_overlap(CANDIDATES, max_per_source=1)
print("Before:", [c["id"] for c in CANDIDATES])
print("After (1 per source):", [c["id"] for c in deduped])

With `max_per_source=1`, only the higher-scoring Alembic chunk (**c2**) remains — **c5** drops unless you raise the limit or merge content at ingest (**04**).

---

## 7. Context Compression

When chunks are too long, **compress** before injection.

| Technique | How it works | Risk |
|-----------|--------------|------|
| **Extractive** | Keep sentences with highest similarity to query | May drop caveats |
| **Abstractive** | Small LLM summarizes chunk conditioned on query | Hallucinate new facts |
| **Token pruning** | LLMLingua-style drop low-salience tokens | Rare tokens lost |
| **Truncate tail** | Hard cut at N tokens | Cuts off conclusions |

Validate compression on the eval set (**09**) — aggressive compression lowers token cost but can remove the one sentence that answers the question.

In [ ]:
QUERY = "alembic upgrade migration"
query_terms = set(QUERY.lower().split())


def extractive_compress(text: str, query_terms: set[str], max_sentences: int = 1) -> str:
    sentences = [s.strip() for s in text.replace("\n", " ").split(".") if s.strip()]
    scored = []
    for s in sentences:
        words = set(s.lower().split())
        overlap = len(query_terms & words)
        scored.append((overlap, s))
    scored.sort(key=lambda x: x[0], reverse=True)
    top = [s for _, s in scored[:max_sentences]]
    return ". ".join(top) + ("." if top else "")


long_chunk = CANDIDATES[1]["text"]
compressed = extractive_compress(long_chunk, query_terms)
print("Original tokens:", count_tokens(long_chunk))
print("Compressed tokens:", count_tokens(compressed))
print("Compressed text:", compressed)

---

## 8. Map-Reduce for Large Evidence Sets

When evidence spans more tokens than one window allows, use **map-reduce**:

```
Many chunks ──map──► partial answer per batch (fits window)
              │
              └──reduce──► final synthesis LLM call
```

| Phase | Input | Output |
|-------|-------|--------|
| **Map** | Batch of chunks + sub-question | Partial notes per batch |
| **Reduce** | All partial notes + original question | Final answer |

Use for "summarize everything in the handbook" or cross-document synthesis — not for simple FAQ lookup where a single packed prompt suffices.

In [ ]:
def map_batches(chunks: list[dict], batch_token_limit: int) -> list[list[dict]]:
    batches, current, used = [], [], 0
    for c in chunks:
        if used + c["tokens"] > batch_token_limit and current:
            batches.append(current)
            current, used = [], 0
        current.append(c)
        used += c["tokens"]
    if current:
        batches.append(current)
    return batches


batches = map_batches(CANDIDATES, batch_token_limit=80)
for i, batch in enumerate(batches, 1):
    print(f"Batch {i}: {[c['id'] for c in batch]}  tokens={sum(c['tokens'] for c in batch)}")

---

## 9. Dynamic $k$ Selection

Fixed `k=5` wastes tokens on easy queries and starves hard ones. **Dynamic $k$** stops adding chunks when the next one exceeds budget.

```python
def dynamic_k(ranked_chunks, budget):
    packed, used = [], 0
    for chunk in ranked_chunks:
        if used + chunk.tokens > budget:
            break
        packed.append(chunk)
        used += chunk.tokens
    return packed
```

| Query type | Typical packed count |
|------------|---------------------|
| Narrow FAQ | 1–2 chunks |
| Multi-part question | 4–8 chunks or map-reduce |
| Unanswerable | 0–1 (trigger abstention **05**) |

---

## 10. Assemble → Format for Prompt (**05**)

Assembly output feeds `format_context_xml` from notebook **05**:

```python
packed = assemble(candidates, budget, order="sandwich")
context = format_context_xml(packed)
messages = build_rag_messages(question, packed)
```

Keep **assembly** separate from **formatting** — unit-test packing without calling the LLM.

In [ ]:
def format_context_xml(chunks: list[dict]) -> str:
    parts = []
    for c in chunks:
        parts.append(f'<doc id="{c["id"]}" source="{c["source"]}">\n{c["text"]}\n</doc>')
    return "\n\n".join(parts)


def assemble_context(
    candidates: list[dict],
    budget: int,
    *,
    dedupe_source: bool = True,
    order: str = "sandwich",
) -> tuple[list[dict], str, int]:
    pool = dedupe_by_source_overlap(candidates) if dedupe_source else candidates
    packed, used = greedy_pack(pool, budget)
    if order == "sandwich":
        packed = sandwich_order(packed)
    return packed, format_context_xml(packed), used


packed, ctx, used = assemble_context(CANDIDATES, budget=180)
print(f"Packed {len(packed)} chunks, {used} tokens")
print(ctx[:400], "...")

---

## 11. Worked Example — Packing Under 600 Tokens

Suppose retrieval returns all five chunks (~940 tokens total) but budget allows **600**:

| Rank | id | Tokens | Action |
|------|-----|--------|--------|
| 1 | c3 | ~45 | Include (sandwich start — JWT) |
| 2 | c2 | ~48 | Include |
| 3 | c5 | ~42 | Skip if deduped with c2 (same source) |
| 4 | c1 | ~40 | Include (sandwich end — API) |
| 5 | c4 | ~38 | Include if budget remains |

Sandwiching **c3** first and **c1** last keeps auth and API overview at attention-favorable positions.

In [ ]:
packed, ctx, used = assemble_context(CANDIDATES, budget=600, dedupe_source=True, order="sandwich")
print("Order:", [c["id"] for c in packed])
print(f"Total tokens used: {used} / 600")

---

## 12. Window Utilization Logging

Per request, log utilization to catch truncation and cost spikes:

| Field | Purpose |
|-------|--------|
| `tokens_system` | Fixed overhead |
| `tokens_context` | Assembled chunks |
| `tokens_history` | Chat turns |
| `tokens_question` | Latest query |
| `utilization_pct` | `prompt_tokens / C_max` |
| `chunks_packed` / `chunks_retrieved` | Assembly efficiency |

Alert when utilization **> 90%** — likely truncation or quality degradation.

In [ ]:
def utilization_report(
    *,
    c_max: int,
    t_system: int,
    t_history: int,
    t_question: int,
    t_context: int,
    t_output: int,
    chunks_retrieved: int,
    chunks_packed: int,
) -> dict:
    prompt_total = t_system + t_history + t_question + t_context
    return {
        "c_max": c_max,
        "prompt_tokens": prompt_total,
        "utilization_pct": round(100 * prompt_total / c_max, 1),
        "context_tokens": t_context,
        "chunks_retrieved": chunks_retrieved,
        "chunks_packed": chunks_packed,
        "headroom": c_max - prompt_total - t_output,
    }


_, _, t_ctx = assemble_context(CANDIDATES, budget=180)
report = utilization_report(
    c_max=C_MAX,
    t_system=T_SYSTEM,
    t_history=T_HISTORY,
    t_question=T_QUESTION,
    t_context=t_ctx,
    t_output=T_OUTPUT_RESERVE,
    chunks_retrieved=len(CANDIDATES),
    chunks_packed=len(greedy_pack(CANDIDATES, 180)[0]),
)
for k, v in report.items():
    print(f"{k:20s}: {v}")

---

## 13. Truncation and Silent Failure

When prompts exceed the window, APIs may **truncate** input without a clear error — the model never sees dropped chunks.

| Prevention | Detail |
|------------|--------|
| **Pre-flight token count** | Count before `chat.completions.create` |
| **Conservative budget** | Target 70–85% of $C_{\max}$ |
| **Assert packed ≤ budget** | Unit test assembler |
| **Compare estimate vs `usage`** | Calibrate tiktoken per corpus |

Silent truncation looks like a **generation** failure but is an **assembly** bug.

---

## 14. Visualization — Budget vs Chunk Count

How many chunks fit as budget grows (illustrative, fixed per-chunk sizes from demo corpus):

In [ ]:
budgets = np.arange(40, 260, 20)
counts = [len(greedy_pack(CANDIDATES, int(b))[0]) for b in budgets]

plt.figure(figsize=(7, 4))
plt.plot(budgets, counts, marker="o", linewidth=2)
plt.xlabel("Context token budget")
plt.ylabel("Chunks packed (greedy)")
plt.title("More budget → more chunks (until all candidates fit)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 15. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Fixed k regardless of budget | Truncation or wasted tokens | Dynamic packing |
| No dedupe | c2 + c5 both consume budget | Per-source or similarity dedupe |
| Weak chunks in middle only | Lost-in-the-middle misses | Sandwich order |
| `len(text)//4` in prod without calibration | Wrong budget | tiktoken + API usage logs |
| Compress without eval | Drop critical sentence | Validate on golden set (**09**) |
| Map-reduce for simple FAQ | 3× LLM cost | Single-window assembly first |

---

## 16. Debugging Assembly Issues

1. **Log packed chunk ids** — Did the answer chunk make the cut?
2. **Compare packed vs retrieved** — Assembly drop vs retrieval miss
3. **Print token breakdown** — System vs history vs context
4. **Try sandwich reorder** — If right chunk was in middle
5. **Increase budget slightly** — If truncation suspected
6. **Disable dedupe** — If dedupe removed complementary chunks

If all packed chunks are correct but answer is wrong → return to **05** prompting.

---

## 17. Summary

Context assembly turns a **ranked candidate list** into a **token-bounded, well-ordered** context block for the LLM. Budget math reserves space for system, history, question, and output before packing chunks. **Greedy packing**, **sandwich ordering**, **deduplication**, and **compression** manage finite windows; **map-reduce** handles evidence too large for one pass.

Demonstrations used the **c1–c5** corpus with tiktoken (or heuristic) counting, greedy and sandwich assemblers, source deduplication, extractive compression, map batches, and utilization reporting.

Retrieve wide (**07** improves what you retrieve); assemble narrow (this notebook). Prompt carefully (**05**).

Back: **05. Prompting and Context Injection for RAG**. Next: **07. Advanced Retrieval for RAG** — hybrid search, query rewriting, and reranking.